<a href="https://colab.research.google.com/github/abuzar072/Sessions/blob/main/Week2_PracticeSession2_AI_Mood_Tracker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install groq gradio pandas matplotlib


  Using cached groq-1.0.0-py3-none-any.whl.metadata (16 kB)
Using cached groq-1.0.0-py3-none-any.whl (138 kB)


In [4]:
# =========================================
# AI Mood Tracker (Groq + Gradio)
# Google Colab Compatible
# =========================================

import os
import json
import csv
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr
from groq import Groq

# =========================================
# CONFIGURATION
# =========================================

CSV_FILE = "mood_data.csv"
MAX_ENTRIES_PER_DAY = 4

MOOD_LIST = [
    "Happy",
    "Sad",
    "Neutral",
    "Stressed",
    "Anxious",
    "Angry",
    "Excited",
    "Tired"
]

MOOD_EMOJI = {
    "Happy": "😊",
    "Sad": "😢",
    "Neutral": "😐",
    "Stressed": "😣",
    "Anxious": "😟",
    "Angry": "😡",
    "Excited": "🤩",
    "Tired": "😴"
}

# =========================================
# GROQ CLIENT (COLAB SECRETS COMPATIBLE)
# =========================================

def get_groq_client():
    """
    Creates and returns a Groq client.
    API key is read from Google Colab Secrets.
    Secret name: GROQ_API_key
    """
    from google.colab import userdata

    api_key =  userdata.get('GROQ_API')

    if not api_key:
        raise RuntimeError(
            "GROQ_API_key not found.\n"
            "Please add it in Google Colab → Secrets → Enable notebook access."
        )

    return Groq(api_key=api_key)


SYSTEM_PROMPT = f"""
You are an emotion classification engine.

Rules:
- Classify the user's mood into EXACTLY ONE of the following moods:
{MOOD_LIST}

- Return ONLY valid JSON
- No markdown
- No extra text
- No explanations outside JSON

Required JSON format:
{{
  "mood": "<one mood from the list>",
  "confidence": <float between 0 and 1>,
  "reason": "<short explanation>"
}}
"""


def groq_analyze_mood(user_text: str) -> dict:
    client = get_groq_client()

    chat_completion = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_text}
        ],
        temperature=0
    )

    raw_output = chat_completion.choices[0].message.content

    try:
        result = json.loads(raw_output)
        if result["mood"] not in MOOD_LIST:
            raise ValueError("Invalid mood returned by model")
        return result
    except Exception as e:
        raise ValueError(f"Failed to parse Groq response: {e}")


# =========================================
# CSV STORAGE
# =========================================

def initialize_csv():
    if not os.path.exists(CSV_FILE):
        with open(CSV_FILE, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow([
                "timestamp",
                "date",
                "time",
                "user_text",
                "detected_mood",
                "confidence"
            ])


def load_mood_data():
    initialize_csv()
    return pd.read_csv(CSV_FILE)


def save_mood_to_csv(user_text, mood, confidence):
    now = datetime.now()
    with open(CSV_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            now.isoformat(),
            str(now.date()),
            now.strftime("%H:%M:%S"),
            user_text,
            mood,
            confidence
        ])


def entries_today_count():
    df = load_mood_data()
    today = str(datetime.now().date())
    return len(df[df["date"] == today])


# =========================================
# ANALYTICS
# =========================================

def generate_analytics():
    df = load_mood_data()

    if df.empty:
        return None, None, "No mood data yet. Start logging your mood!"

    freq = df["detected_mood"].value_counts()

    happy_pct = round((freq.get("Happy", 0) / len(df)) * 100, 2)
    sad_pct = round((freq.get("Sad", 0) / len(df)) * 100, 2)

    summary_text = (
        f"Most common mood: {freq.idxmax()} {MOOD_EMOJI.get(freq.idxmax(), '')}\n"
        f"Happy: {happy_pct}% | Sad: {sad_pct}%"
    )

    # Bar chart
    plt.figure()
    freq.plot(kind="bar")
    plt.title("Mood Frequency")
    plt.xlabel("Mood")
    plt.ylabel("Count")
    bar_chart = plt.gcf()
    plt.close()

    # Trend chart
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    trend = df.groupby(df["timestamp"].dt.date).size()

    plt.figure()
    trend.plot(marker="o")
    plt.title("Mood Entries Over Time")
    plt.xlabel("Date")
    plt.ylabel("Entries")
    line_chart = plt.gcf()
    plt.close()

    return bar_chart, line_chart, summary_text


# =========================================
# GRADIO CALLBACK
# =========================================

def submit_mood(user_text):
    if not user_text.strip():
        return "❌ Please enter a mood description.", None, None

    if entries_today_count() >= MAX_ENTRIES_PER_DAY:
        return "⚠️ You have reached today's limit of 4 mood entries.", None, None

    try:
        result = groq_analyze_mood(user_text)
        save_mood_to_csv(user_text, result["mood"], result["confidence"])

        emoji = MOOD_EMOJI.get(result["mood"], "")
        output = (
            f"Mood: {result['mood']} {emoji}\n"
            f"Confidence: {round(result['confidence'] * 100, 2)}%\n"
            f"Reason: {result['reason']}"
        )

        bar, line, _ = generate_analytics()
        return output, bar, line

    except Exception as e:
        return f"❌ Error: {e}", None, None


# =========================================
# GRADIO UI
# =========================================

def gradio_app():
    with gr.Blocks(title="AI Mood Tracker") as app:
        gr.Markdown("## 🧠 AI Mood Tracker (Groq + Gradio)")
        gr.Markdown("Log your mood up to **4 times per day** and view analytics.")

        mood_input = gr.Textbox(
            label="How are you feeling right now?",
            placeholder="Describe your mood..."
        )

        submit_btn = gr.Button("Submit Mood")

        result_box = gr.Textbox(label="Result", interactive=False)
        bar_plot = gr.Plot(label="Mood Frequency")
        line_plot = gr.Plot(label="Mood Trend")

        submit_btn.click(
            submit_mood,
            inputs=mood_input,
            outputs=[result_box, bar_plot, line_plot]
        )

    return app


# =========================================
# RUN APPLICATION
# =========================================

if __name__ == "__main__":
    initialize_csv()
    app = gradio_app()
    app.launch(debug=True)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8033cb9a8793be9dac.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://8033cb9a8793be9dac.gradio.live
